In [ ]:
import pandas as pd  # a module which provides the data structures and functions to store and manipulate tables in dataframes
import pydbtools as pydb  # A module which allows SQL queries to be run on the Analytical Platform from Python, see https://github.com/moj-analytical-services/pydbtools
import boto3  # allows you to directly create, update, and delete AWS resources from Python scripts

# sets parameters to view dataframes for tables easier
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 900)
pd.set_option("display.max_colwidth", 200)

In [ ]:
#this is the database we will be extracting from
database = "familyman_derived_live_v4" 

#this extracts the latest snapshot from athena
get_snapshot_date = f"SELECT mojap_snapshot_date from {database}.events_derived order by mojap_snapshot_date desc limit 1"
snapshot_date = str(pydb.read_sql_query(get_snapshot_date)['mojap_snapshot_date'].values[0])

#this is the athena database we will be storing our tables in
fcsq_database = "fcsq"

#this is the s3 bucket we will be saving data to
s3 = boto3.resource("s3")
bucket = s3.Bucket("alpha-family-data")

#change these to the current quarter and year not the quarter being published
latest_quarter = 3
latest_year = 2023

#### Extract applicant gender 

In [ ]:
apps_gender = f"""
SELECT
  case_number,
  gender
FROM
  {database}.people_derived
WHERE
  role_model IN ('APLC', 'APLA')
  AND mojap_snapshot_date = date'{snapshot_date}'
"""

pydb.create_temp_table(apps_gender,'apps_gender')

#### Extract respondent data

In [ ]:
resps_gender = f"""
SELECT
  case_number,
  gender
FROM
  {database}.people_derived
WHERE
  role_model IN ('RSPA', 'RSPC', 'RSPZ')
  AND mojap_snapshot_date = date'{snapshot_date}'
"""

pydb.create_temp_table(resps_gender,'resps_gender')

#### Add applicant gender to orders table

In [ ]:
dv_apps_gender = f"""
SELECT
  'Applicant' AS party,
  o.description,
  CASE WHEN a.gender = 1 THEN 'Male'
       WHEN a.gender = 2 THEN 'Female'
         ELSE 'Unknown' END 
        AS Gender
FROM
  fcsq.DV_ords_final o
  LEFT JOIN __temp__.apps_gender a
    ON o.case_number = a.case_number
WHERE
  o.year = 2022
  AND o.quarter = 1
"""

pydb.create_temp_table(dv_apps_gender,'dv_apps_gender')

#### Add respondant data to orders table

In [ ]:
dv_resps_gender = f"""
SELECT
  'Respondant' AS party,
  o.description,
  CASE WHEN r.gender = 1 THEN 'Male'
       WHEN r.gender = 2 THEN 'Female'
         ELSE 'Unknown' END 
        AS Gender
FROM
  fcsq.DV_ords_final o
  LEFT JOIN __temp__.resps_gender r
    ON o.case_number = r.case_number
WHERE
  o.year = 2022
  AND o.quarter = 1
"""

pydb.create_temp_table(dv_resps_gender,'dv_resps_gender')

#### Combine applicant and respondant data

In [ ]:
dv_ords_gender = f"""
SELECT
  *
FROM
  __temp__.dv_apps_gender
UNION ALL
SELECT
  *
FROM
  __temp__.dv_resps_gender
  
"""

pydb.create_temp_table(dv_ords_gender,'dv_ords_gender')

In [ ]:
dv_ords_gender_agg = f"""
SELECT
 party, 
 description,
 gender, 
 count(*) AS count 
FROM
  __temp__.dv_ords_gender 
GROUP BY
  party, 
  description, 
  gender
  
"""

pydb.create_temp_table(dv_ords_gender_agg,'dv_ords_gender_agg')

#### Export data

In [ ]:
dv_gender_data = pydb.read_sql_query("select * from __temp__.dv_ords_gender_agg")

In [ ]:
# Exporting the csv
dv_gender_data.to_csv (r's3://alpha-family-data/FOIs/dv_gender_data.csv', header = True, index=False)